# 0. Problem
## 1633. Percentage of Users Attended a Contest — Easy
For each contest, calculate registered users as a percentage of all users. Round to 2 decimals; sort percentage descending, then contest ID ascending.

Official: https://leetcode.com/problems/percentage-of-users-attended-a-contest/

# 1. Setup

In [ ]:
import pandas as pd
users_rows=[(6,"Alice"),(2,"Bob"),(7,"Alex")]
register_rows=[(215,6),(209,2),(208,2),(210,6),(208,6),(209,7),(209,6),(215,7),(208,7),(210,2),(207,2),(210,7)]
users_pd=pd.DataFrame(users_rows,columns=["user_id","user_name"])
register_pd=pd.DataFrame(register_rows,columns=["contest_id","user_id"])
users_pd, register_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
users_spark=spark.createDataFrame(users_rows,["user_id","user_name"])
register_spark=spark.createDataFrame(register_rows,["contest_id","user_id"])
users_spark.createOrReplaceTempView("Users")
register_spark.createOrReplaceTempView("Register")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT r.contest_id, ROUND(COUNT(DISTINCT r.user_id)*100.0/(SELECT COUNT(*) FROM Users),2) AS percentage
FROM Register r
GROUP BY r.contest_id
ORDER BY percentage DESC, r.contest_id ASC
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
total_users=users_pd["user_id"].nunique()
result_pd=(register_pd.groupby("contest_id",as_index=False).agg(registered_users=("user_id","nunique")).assign(percentage=lambda d:(d["registered_users"]*100/total_users).round(2)).drop(columns="registered_users").sort_values(["percentage","contest_id"],ascending=[False,True]).reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
total_users=users_spark.select("user_id").distinct().count()
result_spark=(register_spark.groupBy("contest_id").agg(F.countDistinct("user_id").alias("registered_users")).withColumn("percentage",F.round(F.col("registered_users")*100.0/F.lit(total_users),2)).drop("registered_users").orderBy(F.desc("percentage"),F.asc("contest_id")))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| distinct count | `COUNT(DISTINCT)` | `.nunique()` | `F.countDistinct()` |
| percentage | scalar denominator | scalar denominator | `F.lit(total)` |
| multi-sort | `ORDER BY ... DESC, ... ASC` | `.sort_values(..., ascending=[...])` | `.orderBy(desc, asc)` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Users, Register

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: users_pd, register_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: users_spark, register_spark